# SAFE binary v8 — 보호집단 공정성 보강 (fresh blind v10)

**왜 이 라운드인가.** 배포 대상 v6(`31b33415`)이 공정성 게이트를 넘지 못한다.

| 측정 축 | v4_r1 `3e9c0b80` | v6 `31b33415` | 게이트 |
| --- | --- | --- | --- |
| UnSmile 7집단 | 0.0207 | 0.0414 | 0.10 |
| KOLD 7집단 | 0.0906 | **0.1263** | 0.10 ❌ |
| 장애 도메인 | 0.0474 | 0.0384 | 0.10 |

v6 최저 집단: `gender-female` 0.8288 / `gender-LGBTQ+` 0.8438 / `race-others` 0.8698.
여성·성소수자 대상 혐오를 다른 집단보다 덜 잡는다.

**이 라운드가 v6과 다른 점.** 개발 게이트에 **공정성 조건을 넣는다.** v6은 위험 탐지
게이트(blind v8)만 보고 통과시켰고, 그 사이 형평이 무너졌는데 아무도 못 봤다.
여기서는 재현율 격차가 개선되지 않으면 후보를 승격하지 않는다.

**방법론(이전 라운드와 동일).**
- fresh blind는 후보 고정 후 **1회만** 소비하고 해시로 검증한다.
- 소비된 blind(v1~v9)는 dev 회귀셋으로 격하해 쓴다. train에 병합하지 않는다.
- 규칙 보조(RULE_ASSIST)는 OFF.
- 보강 데이터는 **train split에서만** 뽑는다. 원문을 새로 만들지 않는다.
- blind v10 작성자는 v1~v9와 같이 사용자여도 된다. 이 라운드는 새 문장을 생성하지
  않으므로 자기 채점 위험이 없다. 다만 **학습 전에** 쓰고, 약집단에 몰아 쓰지 않는다.

> **blind v10 주의.** 노트북 13(사기 v7)도 v10을 기대하도록 적혀 있다. 이 라운드가
> 먼저 소비하므로, 사기 라운드를 재개할 때 노트북 13을 v11로 올려야 한다.


In [ ]:
# 1. 런타임·저장소 확인 (재학습은 GPU 필수)
#    Colab VM은 초기화되면 저장소가 사라진다. 없으면 clone 한다.
#    작업 디렉터리가 '/' 인 경우가 있어 cwd 기준 탐색만으로는 못 찾는다.
import subprocess, sys, json, hashlib, re, os, shutil, tempfile, collections
from pathlib import Path
import torch

assert torch.cuda.is_available(), '재학습은 GPU 런타임에서만 실행할 수 있습니다.'
print('GPU:', torch.cuda.get_device_name(0))

BRANCH = 'feature/safe-scam-augmentation'
REMOTE = 'https://github.com/threeGuineas/thisabled-ai.git'

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, cwd / 'thisabled-ai',
              Path('/content/thisabled-ai'), Path.home() / 'thisabled-ai']
REPO = next((p for p in candidates if (p / '.git').is_dir()), None)
if REPO is None:
    clone_parent = Path('/content') if Path('/content').is_dir() else cwd
    REPO = clone_parent / 'thisabled-ai'
    assert not REPO.exists(), f'clone 대상이 이미 있으나 git 저장소가 아닙니다: {REPO}'
    print(f'저장소가 없어 clone 합니다 -> {REPO}')
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REMOTE, str(REPO)],
                   check=True)
assert (REPO / '.git').is_dir(), f'저장소 준비 실패: {REPO}'

current = subprocess.check_output(['git', 'branch', '--show-current'], cwd=REPO, text=True).strip()
assert current == BRANCH, f'현재 브랜치 {current!r} != {BRANCH!r}'
subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'],
               cwd=REPO, check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)          # 이후 셀의 상대 경로 기준을 저장소 루트로 고정

print('repo  :', REPO)
print('commit:', subprocess.check_output(['git', 'log', '--oneline', '-1'], cwd=REPO, text=True).strip())

CONFIG = REPO / 'configs/module1_binary_hardcases_v8.yaml'
assert CONFIG.exists(), 'configs/module1_binary_hardcases_v8.yaml 없음 — pull 확인'
print('config:', CONFIG.relative_to(REPO))


In [ ]:
# 2. 평가 데이터 번들 반영
#    data/eval/* 는 라이선스상 재배포 금지라 저장소에 없다. clone 직후엔 반드시 필요하다.
#    주의: 이 환경(VS Code + Colab 확장)에서는 google.colab files.upload / drive.mount
#    위젯이 동작하지 않는다. 아래 둘 중 하나로 전달한다.
#      (A) VS Code 탐색기로 data_bundle.zip 을 저장소 루트에 올린 뒤 이 셀 실행
#      (B) BUNDLE_GDRIVE_ID 에 개인 Drive 파일 ID 를 넣고 실행 (gdown 사용)
import zipfile

BUNDLE_GDRIVE_ID = ''      # 예: '1AbCdEf...' — (B) 방식일 때만 채운다

REQUIRED = ['data/eval/aihub_train.jsonl',
            'data/eval/aihub_real_holdout.jsonl',
            'data/eval/beep_real_holdout.jsonl',
            'data/synthetic/pan12_translated.jsonl']

missing = [rel for rel in REQUIRED if not (REPO / rel).exists()]
if not missing:
    print('필요 파일 모두 존재 — 번들 반영 생략')
else:
    print('없는 파일:', missing)
    search = [REPO / 'data_bundle.zip', Path.cwd() / 'data_bundle.zip',
              Path('/content/data_bundle.zip'), Path.home() / 'data_bundle.zip']
    src = next((p for p in search if p.exists()), None)

    if src is None and BUNDLE_GDRIVE_ID:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
        src = REPO / 'data_bundle.zip'
        subprocess.run([sys.executable, '-m', 'gdown', '--id', BUNDLE_GDRIVE_ID,
                        '-O', str(src)], check=True)

    assert src is not None, (
        'data_bundle.zip 을 찾지 못했습니다.\n'
        f'  (A) VS Code 탐색기로 {REPO}/data_bundle.zip 에 올린 뒤 이 셀을 다시 실행하거나\n'
        '  (B) 위 BUNDLE_GDRIVE_ID 에 Drive 파일 ID 를 넣고 다시 실행하세요.\n'
        f'  찾아본 경로: {[str(p) for p in search]}')

    print('번들:', src)
    with zipfile.ZipFile(src) as zf:
        for member in zf.namelist():
            if member.endswith('/'):
                continue
            target = (REPO / member).resolve()
            assert str(target).startswith(str(REPO.resolve())), f'zip 경로 이탈: {member}'
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(member) as fh, target.open('wb') as out:
                shutil.copyfileobj(fh, out)

    still = [rel for rel in REQUIRED if not (REPO / rel).exists()]
    assert not still, f'번들 반영 후에도 없음: {still}'
    print('번들 반영 완료')

for rel in REQUIRED:
    path = REPO / rel
    print(f'  {rel:<45}{path.stat().st_size:>10,} bytes')


In [ ]:
# 3. fresh blind v10 존재·봉인 확인 (소비는 셀 8에서 1회)
#    v1~v9와 마찬가지로 사용자가 직접 작성한다. 이 라운드는 기존 train 행을
#    반복할 뿐 새 문장을 만들지 않으므로 자기 채점 문제가 없다.
#    단, 반드시 학습 전에 작성하고 이후 수정하지 않는다(아래 해시로 고정).
BLIND = REPO / 'tests/fixtures/safe_blind_v10.jsonl'
assert BLIND.exists(), (
    'fresh blind v10 없음. tests/fixtures/safe_blind_v10.jsonl 을 먼저 작성한다 '
    '(40행, v5~v9와 동일 스키마: text/label/slice/receiver_is_minor).')

blind_rows = [json.loads(x) for x in BLIND.read_text(encoding='utf-8').splitlines() if x.strip()]
BLIND_SHA = hashlib.sha256(BLIND.read_bytes()).hexdigest()
label_counts = collections.Counter(int(r['label']) for r in blind_rows)
slice_counts = collections.Counter(str(r['slice']) for r in blind_rows)

print(f'blind v10  {len(blind_rows)}행  sha256 {BLIND_SHA}')
print('  라벨 분포 :', dict(sorted(label_counts.items())))
print('  슬라이스  :', dict(sorted(slice_counts.items())))
assert len(blind_rows) >= 40, 'blind 가 너무 작다'
assert label_counts[0] > 0 and label_counts[1] > 0, 'blind 에 두 라벨이 모두 있어야 한다'
assert all('receiver_is_minor' in r for r in blind_rows), 'receiver_is_minor 누락'


In [ ]:
# 4. 데이터 빌드 + 약집단 오버샘플 생성 + 누수 가드
#    보강 데이터는 train split 에서만 뽑는다. 새 문장을 만들지 않으므로 라벨 노이즈가 없다.
import pandas as pd
import numpy as np

BASELINE_FAIRNESS = REPO / 'artifacts/safe_binary_fairness_v6_31b33415.json'
OVERSAMPLE_MULTIPLIER = 3      # 약집단 train 행 반복 횟수
RECALL_FLOOR = 0.92            # 이 값 미만인 집단을 약집단으로 본다

steps = [
    [sys.executable, 'scripts/download_seed_datasets.py'],
    [sys.executable, 'scripts/build_processed_dataset.py'],
    [sys.executable, 'scripts/build_final_dataset.py', '--synth-repeat', '1', '--include-aihub-train'],
    [sys.executable, 'scripts/build_safe_hardcase_dataset.py', '--include-v5',
     '--output', 'data/synthetic/safe_hardcases_v5/train.jsonl',
     *sum([['--forbidden', f'tests/fixtures/safe_blind_v{i}.jsonl'] for i in range(1, 11)], [])],
    [sys.executable, 'scripts/adapt_external_datasets.py'],
]
for cmd in steps:
    subprocess.run(cmd, cwd=REPO, check=True)

train = pd.read_parquet(REPO / 'data/processed/train.parquet')
test = pd.read_parquet(REPO / 'data/processed/test.parquet')
val = pd.read_parquet(REPO / 'data/processed/val.parquet')
train['label_bin'] = (train['label'] > 0).astype(int)

# --- 약집단 식별: 기준선 공정성 JSON 의 recall 이 낮은 집단 ---
baseline = json.loads(BASELINE_FAIRNESS.read_text(encoding='utf-8'))
weak = {'kold': [], 'unsmile': []}
for group, row in baseline['kold_top_groups']['groups'].items():
    if row.get('recall') is not None and row['recall'] < RECALL_FLOOR:
        weak['kold'].append(group)
for group, row in baseline['unsmile_7_groups']['groups'].items():
    if row.get('recall') is not None and row['recall'] < RECALL_FLOOR:
        weak['unsmile'].append(group)
print('약집단(KOLD)   :', weak['kold'])
print('약집단(UnSmile):', weak['unsmile'])
assert weak['kold'] or weak['unsmile'], '약집단이 없다 — RECALL_FLOOR 를 확인하라'

# --- train split 에서 해당 집단 행 추출 ---
kold_raw = pd.DataFrame(json.loads((REPO / 'data/raw/kold/kold_v1.json').read_text(encoding='utf-8')))
kold_grp = kold_raw.set_index('guid')['GRP'].to_dict()
unsmile_raw = pd.concat([
    pd.read_csv(REPO / 'data/raw/unsmile/unsmile_train_v1.0.tsv', sep='\t'),
    pd.read_csv(REPO / 'data/raw/unsmile/unsmile_valid_v1.0.tsv', sep='\t'),
], ignore_index=True)

picked = []
kold_train = train[train['source'] == 'kold'].copy()
kold_train['grp'] = kold_train['source_id'].map(kold_grp)
picked.append(kold_train[kold_train['grp'].isin(weak['kold'])])

if weak['unsmile']:
    us_train = train[train['source'] == 'unsmile'].copy()
    idx = us_train['source_id'].astype(int)
    mask = np.zeros(len(us_train), dtype=bool)
    for group in weak['unsmile']:
        flags = unsmile_raw[group].reindex(idx.values).fillna(0).to_numpy()
        mask |= (flags == 1)
    picked.append(us_train[mask])

weak_rows = pd.concat(picked, ignore_index=True).drop_duplicates(subset=['text'])
print(f'\n약집단 train 행 {len(weak_rows)}건  라벨 {weak_rows["label_bin"].value_counts().sort_index().to_dict()}')
assert len(weak_rows) > 0, '약집단 train 행이 없다'


In [ ]:
# 5. 오버샘플 파일 기록 + 누수 검증 (val/test/blind v1~v10 과 교차 0 이어야 한다)
def norm(x):
    return re.sub(r'[^0-9a-z가-힣]+', '', str(x).lower())

forbidden_texts = set()
for frame in (val, test):
    forbidden_texts |= {norm(t) for t in frame['text']}
for i in range(1, 11):
    path = REPO / 'tests/fixtures' / f'safe_blind_v{i}.jsonl'
    if path.exists():
        forbidden_texts |= {norm(json.loads(x)['text'])
                            for x in path.read_text(encoding='utf-8').splitlines() if x.strip()}
for name in ('aihub_real_holdout.jsonl', 'beep_real_holdout.jsonl'):
    forbidden_texts |= {norm(json.loads(x)['text'])
                        for x in (REPO / 'data/eval' / name).read_text(encoding='utf-8').splitlines() if x.strip()}

leaked = [t for t in weak_rows['text'] if norm(t) in forbidden_texts]
print(f'누수 검사: 약집단 {len(weak_rows)}건 중 평가셋과 겹침 {len(leaked)}건')
assert not leaked, f'누수 {len(leaked)}건 — 오버샘플 중단. 예: {leaked[:2]}'

out_dir = REPO / 'data/synthetic/fairness_v8'
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'train.jsonl'
records = []
for _, row in weak_rows.iterrows():
    for _ in range(OVERSAMPLE_MULTIPLIER):
        records.append({'text': row['text'], 'label': int(row['label_bin']),
                        'source': 'fairness_oversample_v8'})
with out_path.open('w', encoding='utf-8') as fh:
    for record in records:
        fh.write(json.dumps(record, ensure_ascii=False) + '\n')

FAIRNESS_SHA = hashlib.sha256(out_path.read_bytes()).hexdigest()
print(f'기록: {out_path.relative_to(REPO)}  {len(records)}행 (고유 {len(weak_rows)} x {OVERSAMPLE_MULTIPLIER})')
print('sha256', FAIRNESS_SHA)
print('라벨 분포:', collections.Counter(r['label'] for r in records))


In [ ]:
# 6. 개발 평가 — dev 회귀셋(실 holdout + 소비 blind v1~v9) + **공정성 조건**
#    blind v10 은 여기서 읽지 않는다. 규칙 보조 OFF.
from sklearn.metrics import confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from src.evaluation.fairness import run_full_fairness_evaluation

real = []
for name in ('aihub_real_holdout.jsonl', 'beep_real_holdout.jsonl'):
    real += [json.loads(x) for x in (REPO / 'data/eval' / name).read_text(encoding='utf-8').splitlines() if x.strip()]
dev = []
for i in range(1, 10):
    path = REPO / 'tests/fixtures' / f'safe_blind_v{i}.jsonl'
    if path.exists():
        dev += [json.loads(x) for x in path.read_text(encoding='utf-8').splitlines() if x.strip()]
print(f'dev 회귀셋 {len(dev)}행 / 실 holdout {len(real)}행')

def load_predict(path):
    tok = AutoTokenizer.from_pretrained(path)
    model = AutoModelForSequenceClassification.from_pretrained(path).cuda().eval()
    assert model.config.num_labels == 2
    def predict(texts, batch=128):
        out = []
        with torch.inference_mode():
            for i in range(0, len(texts), batch):
                enc = tok(texts[i:i + batch], padding=True, truncation=True,
                          max_length=128, return_tensors='pt').to('cuda')
                out.extend(torch.softmax(model(**enc).logits, dim=-1)[:, 1].cpu().tolist())
        return np.array(out)
    return predict

FAIRNESS_GATE = 0.10
V6_KOLD_GAP = baseline['kold_top_groups']['max_recall_gap']

def evaluate_fairness_at(predict, adult, minor):
    """test.parquet 기준 집단별 재현율 격차. 배포 판정과 같은 축이다."""
    frame = test.reset_index(drop=True).copy()
    frame['label'] = (frame['label'] > 0).astype(int)
    probs = predict(frame['text'].tolist())
    frame_minor = frame.get('receiver_is_minor')
    thr = np.full(len(frame), adult) if frame_minor is None else np.where(frame_minor, minor, adult)
    y_pred = (probs >= thr).astype(int)
    result = run_full_fairness_evaluation(
        test_df=frame, y_pred=y_pred,
        unsmile_raw_train_path=REPO / 'data/raw/unsmile/unsmile_train_v1.0.tsv',
        unsmile_raw_valid_path=REPO / 'data/raw/unsmile/unsmile_valid_v1.0.tsv',
        kold_raw_path=REPO / 'data/raw/kold/kold_v1.json')
    return {axis: result[axis]['max_recall_gap']
            for axis in ('unsmile_7_groups', 'kold_top_groups', 'disability_domain')}, result

def evaluate_dev(path):
    predict = load_predict(path)
    yr = np.array([int(int(x['label']) > 0) for x in real]); pr = predict([x['text'] for x in real])
    yd = np.array([int(x['label']) for x in dev]); pdv = predict([x['text'] for x in dev])
    best = None
    for adult in np.arange(0.40, 0.86, 0.01):
        minor = max(0.35, round(float(adult) - 0.16, 2))
        rpred = (pr >= adult)
        dthr = np.array([minor if x['receiver_is_minor'] else adult for x in dev])
        dpred = (pdv >= dthr)
        tn, fp, fn, tp = confusion_matrix(yr, rpred, labels=[0, 1]).ravel()
        dtn, dfp, dfn, dtp = confusion_matrix(yd, dpred, labels=[0, 1]).ravel()
        real_recall = tp / max(tp + fn, 1); real_spec = tn / max(tn + fp, 1)
        dev_recall = dtp / max(dtp + dfn, 1); dev_spec = dtn / max(dtn + dfp, 1)
        if real_recall < 0.80 or real_spec < 0.85 or dev_recall < 0.90 or dev_spec < 0.90:
            continue
        score = real_recall + real_spec + dev_recall + dev_spec
        if best is None or score > best['score']:
            best = {'adult': round(float(adult), 2), 'minor': round(float(minor), 2),
                    'score': float(score), 'real_recall': float(real_recall),
                    'real_specificity': float(real_spec), 'dev_recall': float(dev_recall),
                    'dev_specificity': float(dev_spec)}
    if best is None:
        return {'pass': False, 'reason': '운영점을 찾지 못함'}

    gaps, detail = evaluate_fairness_at(predict, best['adult'], best['minor'])
    best['fairness_gaps'] = gaps
    best['fairness_detail'] = {k: detail[k]['groups'] for k in gaps}
    # 이 라운드의 목적: KOLD 격차가 게이트 아래로 내려오고 v6 보다 개선돼야 한다.
    best['pass'] = (gaps['kold_top_groups'] <= FAIRNESS_GATE
                    and gaps['unsmile_7_groups'] <= FAIRNESS_GATE
                    and gaps['disability_domain'] <= FAIRNESS_GATE
                    and gaps['kold_top_groups'] < V6_KOLD_GAP)
    return best

print(f'공정성 게이트 {FAIRNESS_GATE} / v6 KOLD 격차 {V6_KOLD_GAP:.4f} 보다 낮아야 통과')


In [ ]:
# 7. v8 재학습 repeat 1→3. 개발 게이트(위험 + 공정성) 통과 시 즉시 중단
import yaml

SELECTED = None
for repeat in (1, 2, 3):
    cfg = yaml.safe_load(CONFIG.read_text(encoding='utf-8'))
    cfg['data']['extra_train_repeat'] = repeat
    ckpt = f"models/checkpoints/module1_binary_hardcases_v8_r{repeat}"
    cfg['model']['checkpoint_dir'] = ckpt
    cfg['paths']['checkpoint_dir'] = ckpt
    tmp = REPO / f'configs/.v8_r{repeat}.yaml'
    tmp.write_text(yaml.safe_dump(cfg, allow_unicode=True), encoding='utf-8')
    print(f'\n=== repeat {repeat} 학습 ===')
    subprocess.run([sys.executable, 'scripts/train_module1.py', '--config', str(tmp.relative_to(REPO))],
                   cwd=REPO, check=True)
    tmp.unlink()

    result = evaluate_dev(REPO / ckpt)
    result['repeat'] = repeat
    result['checkpoint'] = str(REPO / ckpt)
    print(json.dumps({k: v for k, v in result.items() if k != 'fairness_detail'},
                     ensure_ascii=False, indent=2))
    if result.get('pass'):
        SELECTED = result
        print(f'\n개발 게이트 통과 — repeat {repeat} 에서 중단')
        break

assert SELECTED is not None, (
    '3회 모두 개발 게이트 미통과. OVERSAMPLE_MULTIPLIER 를 올리거나 '
    '약집단 선정 기준(RECALL_FLOOR)을 넓혀 재시도한다.')
SELECTED_SHA = hashlib.sha256(
    (Path(SELECTED['checkpoint']) / 'model.safetensors').read_bytes()).hexdigest()
print('checkpoint sha256', SELECTED_SHA)


In [ ]:
# 8. 후보 고정 후 fresh blind v10 최초 1회 평가 (규칙 보조 OFF)
assert SELECTED is not None
assert hashlib.sha256(BLIND.read_bytes()).hexdigest() == BLIND_SHA, 'blind v10 이 셀 3 이후 변경됨'
assert hashlib.sha256((Path(SELECTED['checkpoint']) / 'model.safetensors').read_bytes()).hexdigest() == SELECTED_SHA, \
    '선택 checkpoint 가 변경됨'

predict = load_predict(SELECTED['checkpoint'])
adult, minor = SELECTED['adult'], SELECTED['minor']
texts = [r['text'] for r in blind_rows]
y_true = np.array([int(r['label']) for r in blind_rows])
probs = predict(texts)
thr = np.array([minor if r['receiver_is_minor'] else adult for r in blind_rows])
y_pred = (probs >= thr).astype(int)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
risk_recall = tp / max(tp + fn, 1)
specificity = tn / max(tn + fp, 1)
macro_f1 = 0.5 * ((2 * tp / max(2 * tp + fp + fn, 1)) + (2 * tn / max(2 * tn + fn + fp, 1)))
by_slice = {}
for name in sorted({str(r['slice']) for r in blind_rows if int(r['label']) == 1}):
    idx = [i for i, r in enumerate(blind_rows) if str(r['slice']) == name and int(r['label']) == 1]
    by_slice[name] = float(y_pred[idx].mean()) if idx else None

blind_result = {
    'blind': 'safe_blind_v10.jsonl', 'sha256': BLIND_SHA,
    'thresholds': {'adult': adult, 'minor': minor}, 'rule_assist': False,
    'confusion_matrix': [[int(tn), int(fp)], [int(fn), int(tp)]],
    'risk_recall': float(risk_recall), 'specificity': float(specificity),
    'macro_f1': float(macro_f1), 'risk_slices': by_slice,
    'checkpoint_sha256': SELECTED_SHA,
}
print(json.dumps(blind_result, ensure_ascii=False, indent=2))

out = REPO / 'outputs/10_thisabled-ai/보고서'
out.mkdir(parents=True, exist_ok=True)
(out / 'safe_blind_v10_결과.json').write_text(
    json.dumps(blind_result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('\n소비 기록 저장 — blind v10 은 이제 dev 회귀셋으로 격하된다.')


In [ ]:
# 9. 최종 공정성 재측정 + v4/v6 대비 (이 라운드의 목적 지표)
gaps, detail = evaluate_fairness_at(predict, adult, minor)

rows = [('v4_r1 3e9c0b80', {'unsmile_7_groups': 0.0207, 'kold_top_groups': 0.0906,
                            'disability_domain': 0.0474}),
        ('v6    31b33415', {axis: baseline[axis]['max_recall_gap']
                            for axis in gaps}),
        ('v8    (이번)', gaps)]
print(f"{'모델':<18}{'unsmile':>10}{'kold':>10}{'장애':>10}")
for name, g in rows:
    print(f"{name:<18}{g['unsmile_7_groups']:>10.4f}{g['kold_top_groups']:>10.4f}"
          f"{g['disability_domain']:>10.4f}")

print('\n=== v8 KOLD 집단별 위험 재현율 ===')
for group, row in sorted(detail['kold_top_groups']['groups'].items(),
                         key=lambda kv: (kv[1].get('recall') is None, kv[1].get('recall'))):
    if row.get('recall') is not None:
        print(f"  {group:<24}{row['n']:>5}건  {row['recall']:.4f}")

fairness_payload = {'model': {'checkpoint': SELECTED['checkpoint'], 'sha256': SELECTED_SHA,
                              'thresholds': {'adult': adult, 'minor': minor}},
                    'gaps': gaps, 'detail': detail, 'blind_v10': blind_result}
(REPO / 'artifacts').mkdir(exist_ok=True)
(REPO / 'artifacts/safe_binary_fairness_v8_candidate.json').write_text(
    json.dumps(fairness_payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('\nsaved: artifacts/safe_binary_fairness_v8_candidate.json')


In [ ]:
# 10. 오판 확인 — FN(놓친 위험) / FP(과플래그)
for i, row in enumerate(blind_rows):
    truth, pred = int(row['label']), int(y_pred[i])
    if truth == pred:
        continue
    kind = 'FN' if truth == 1 else 'FP'
    print(f"{kind} p={probs[i]:.4f} [{row['slice']}] {row['text'][:70]}")


In [ ]:
# 11. 명시적으로 켠 경우에만 HF 업로드 (추론 파일만)
UPLOAD_TO_HF = False
HUMAN_APPROVED = False      # 위 수치를 직접 확인한 뒤에만 True

if UPLOAD_TO_HF:
    assert HUMAN_APPROVED, '사람 확인 없이 업로드하지 않는다.'
    assert SELECTED is not None and SELECTED.get('pass'), '게이트 미통과 후보는 올리지 않는다.'
    assert gaps['kold_top_groups'] <= FAIRNESS_GATE, '공정성 게이트 미달 — 업로드 금지'
    from getpass import getpass
    from huggingface_hub import login, upload_folder
    token = getpass('HF write token: ')
    login(token=token, add_to_git_credential=False); del token
    url = upload_folder(
        repo_id='soyuncj/thisabled-safety-kcelectra',
        folder_path=SELECTED['checkpoint'],
        commit_message=f"retrain v8 fairness r{SELECTED['repeat']} blind-v10 approved "
                       f"(kold gap {gaps['kold_top_groups']:.4f})",
        ignore_patterns=['checkpoint-*', 'optimizer*', 'scheduler*',
                         'trainer_state*', 'rng_state*', 'training_args*'])
    print('HF_COMMIT_URL:', url)
else:
    print('검증 완료. 업로드는 비활성 상태입니다.')
